# Encoder timing

Reproduces the encoder columns of Table I: Params., Dim., Input prep., Feat. ext. and Total.

Thirty subjects are drawn with seed 0 from the NKI subjects of the main benchmark, and each model receives its main benchmark input: the affine TurboPrep volume, or the CAT12 gray matter map for AnatCL, which runs its five global descriptor fold models and averages their features. Execution is restricted to one CPU thread. For each model, loading the weights and one warm up pass over the 30 subjects are not timed. Each subject then goes through two stages timed with `time.perf_counter`:

- input preparation: loading the volume, resizing or cropping it to the model grid and normalizing intensities, plus the rank filtered and Sobel channels for the un-CNN;
- feature extraction: one forward pass under `torch.no_grad` returning the feature vector.

The 30 subject pass is repeated 10 times, and the table reports the mean and standard deviation across repetitions of the per image mean. The paper run used one SLURM job with 2 CPUs and 16 GB, submitted by `slurm/encoder_timing.sh`.

In [ ]:
N_SUBJECTS = 30
SEED = 0
REPS = 10
MODELS = ["Untrained_CNN", "3D-Neuro-SimCLR", "AnatCL_Global", "SwinBrain", "BrainIAC"]

In [ ]:
import os

for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"

import gc
import time

import numpy as np
import pandas as pd
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

import cohorts
import encoders
from config import OUTPUT_DIR
from machine import system_info

OUT = OUTPUT_DIR / "encoder_timing"
OUT.mkdir(parents=True, exist_ok=True)
info = system_info()
pd.DataFrame([info]).to_csv(OUT / "system_info.csv", index=False)
pd.Series(info)

## Subjects

In [ ]:
nki = cohorts.load("NKI")
chosen = sorted(np.random.default_rng(SEED).choice(nki.subjects, N_SUBJECTS, replace=False))
pos = {s: i for i, s in enumerate(nki.subjects)}
inputs = {"affine": [(s, nki.affine[pos[s]]) for s in chosen], "cat12": [(s, nki.cat12[pos[s]]) for s in chosen]}
pd.DataFrame({"subject": chosen, "affine": [f for _, f in inputs["affine"]], "cat12": [f for _, f in inputs["cat12"]]}
             ).to_csv(OUT / "subjects.csv", index=False)

## Timing

In [ ]:
raw, meta = [], []
for name in MODELS:
    load, prepare, _, kind = encoders.ENCODERS[name]
    gc.collect()
    model = load()
    with torch.no_grad():
        dim = max(model(prepare(f)[None]).reshape(1, -1).shape[1] for _, f in inputs[kind])
    for rep in range(REPS):
        for sid, f in inputs[kind]:
            t0 = time.perf_counter()
            x = prepare(f)[None]
            t1 = time.perf_counter()
            with torch.no_grad():
                model(x).detach().cpu().numpy().reshape(1, -1)
            t2 = time.perf_counter()
            raw.append({"model": name, "rep": rep, "subject": sid, "prep_s": t1 - t0, "feat_s": t2 - t1, "total_s": t2 - t0})
        print(f"{name:<16} rep {rep + 1}/{REPS}: {np.mean([r['total_s'] for r in raw[-len(inputs[kind]):]]):.3f} s/img")
    meta.append({"model": name, "input": kind, "n_params": encoders.n_params(model), "dim": dim})
    del model

raw = pd.DataFrame(raw)
raw.to_csv(OUT / "raw.csv", index=False)

## Table I

Total adds the image preprocessing time that produces the model input: 1 to 2 minutes per image for TurboPrep, as given in the preprocessing section of the paper, and 7 to 8 minutes for CAT12, from the interquartile range, 7.25 to 8.05 minutes, of the volume processing time in the CAT12 logs of the NKI subjects. Neither was measured here.

In [ ]:
per_rep = raw.groupby(["model", "rep"])[["prep_s", "feat_s", "total_s"]].mean()
agg = per_rep.groupby("model").agg(["mean", "std"])
agg.columns = [f"{c}_{s}" for c, s in agg.columns]
summary = pd.DataFrame(meta).set_index("model").join(agg)
summary.to_csv(OUT / "summary.csv")

PREPROC_MIN = {"affine": (1, 2), "cat12": (7, 8)}
table = pd.DataFrame({
    "Params.": summary["n_params"].map(lambda n: f"{n / 1e6:.2f}M"),
    "Dim.": summary["dim"],
    "Input prep. (s/img)": summary.apply(lambda r: f"{r.prep_s_mean:.3f} ± {r.prep_s_std:.3f}", axis=1),
    "Feat. ext. (s/img)": summary.apply(lambda r: f"{r.feat_s_mean:.3f} ± {r.feat_s_std:.3f}", axis=1),
    "Encoder (s/img)": summary.apply(lambda r: f"{r.total_s_mean:.2f} ± {r.total_s_std:.2f}", axis=1),
    "Total (m/img)": summary.apply(lambda r: "{:.1f}–{:.1f}".format(*(m + r.total_s_mean / 60 for m in PREPROC_MIN[r.input])), axis=1),
})
table